# Data & Feature Sanity Checks

This notebook validates the raw dataset and the engineered feature matrix before any model
training. Run top-to-bottom: every assertion raises `AssertionError` on failure.

Each section corresponds to a specific claim made in the paper's EDA or methodology sections.
Passing all assertions confirms that the implementation matches the documented rationale.

## What this notebook checks

| Section | What is validated | Paper claim |
|---|---|---|
| 1 | Shape, nulls, column naming | 5,000 rows, no missing values |
| 2 | Class balance per target | IncomeInvestment ~38%, AccumulationInvestment ~51% |
| 3 | Skewness of Wealth and Income | Log transforms are justified |
| 4 | Engineered feature formulas and exclusions | F_E definition, Gender/FamilyMembers excluded |
| 5 | Scaler fitted on train only | No data leakage |
| 6 | Class ratio preserved across split | Stratified split working for both targets |
| 7 | Feature-target and inter-feature correlations | All 7 quantitative EDA claims |
| 8 | Feature set structure and composition | F_E vs F_B diff is exactly as designed |
| 9 | Model output directory status | Pre-training state |

In [30]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("sanity_checks")

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import skew

# Make utils importable from the notebook
sys.path.insert(0, str(Path().resolve()))

from utils.preprocessing import (
    BASELINE_FEATURE_NAMES,
    FEATURE_NAMES,
    TARGETS,
    build_baseline_features,
    build_features,
    load_data,
    split_and_scale,
)

## 1. Raw data — shape, types, missing values

In [2]:
df = load_data()
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nFirst rows:")
df.head()

Shape: (5000, 9)

Dtypes:
Age                         int64
Gender                      int64
FamilyMembers               int64
FinancialEducation        float64
RiskPropensity            float64
Income                    float64
Wealth                    float64
IncomeInvestment            int64
AccumulationInvestment      int64
dtype: object

First rows:


,Age,Gender,FamilyMembers,FinancialEducation,RiskPropensity,Income,Wealth,IncomeInvestment,AccumulationInvestment
0,60,0,2,0.228685,0.233355,68.181525,53.260067,0,1
1,78,0,2,0.358916,0.170911,21.807595,135.550048,1,0
2,33,1,2,0.317515,0.249703,23.252747,66.303678,0,1
3,69,1,4,0.767685,0.654597,166.189034,404.997689,1,1
4,58,0,3,0.429719,0.349039,21.186723,58.911930,0,0


In [31]:
assert df.shape[0] == 5000, f"Expected 5000 rows, got {df.shape[0]}"
null_count = df.isnull().sum().sum()
assert null_count == 0, f"Found {null_count} null values"

assert 'Income' in df.columns, "'Income' column not found — strip() may have failed"
assert 'Income ' not in df.columns, "Trailing-space bug in 'Income ' still present"

logger.info("Section 1 PASSED — shape %s, nulls %d, Income column clean", df.shape, null_count)

14:50:28  INFO      Section 1 PASSED — shape (5000, 9), nulls 0, Income column clean


## 2. Class balance

In [32]:
for target in TARGETS:
    prevalence = df[target].mean()
    counts = df[target].value_counts().to_dict()
    print(f"{target}: prevalence={prevalence:.3f}  counts={counts}")

# IncomeInvestment expected ~38% positive
income_prev = df['IncomeInvestment'].mean()
assert 0.35 < income_prev < 0.42, f"IncomeInvestment prevalence={income_prev:.3f} outside [0.35, 0.42]"

# AccumulationInvestment expected ~51% positive
accum_prev = df['AccumulationInvestment'].mean()
assert 0.48 < accum_prev < 0.54, f"AccumulationInvestment prevalence={accum_prev:.3f} outside [0.48, 0.54]"

logger.info("Class balance checks PASSED")

14:50:42  INFO      Class balance checks PASSED


IncomeInvestment: prevalence=0.384  counts={0: 3082, 1: 1918}
AccumulationInvestment: prevalence=0.513  counts={1: 2566, 0: 2434}


## 3. Skewness — confirms log transform necessity

In [33]:
SKEW_THRESHOLDS = {
    'Wealth': 2.0,  # severe power-law tail
    'Income': 1.0,  # moderate right skew — box plot confirms outliers at 4-6x the mean
}

for col, threshold in SKEW_THRESHOLDS.items():
    sk = skew(df[col])
    print(f"{col}: skewness = {sk:.2f}  (threshold > {threshold})")
    assert sk > threshold, \
        f"{col} skewness={sk:.2f} is not high enough to justify log transform"

logger.info("Skewness checks PASSED — log transforms are justified")

14:50:53  INFO      Skewness checks PASSED — log transforms are justified


Wealth: skewness = 5.83  (threshold > 2.0)
Income: skewness = 1.38  (threshold > 1.0)


## 4. Engineered feature set — structure and formula spot-checks

In [35]:
X = build_features(df)
logger.info("Engineered features shape: %s", X.shape)
logger.info("Columns: %s", list(X.columns))
X.describe()

14:51:15  INFO      Engineered features shape: (5000, 10)
14:51:15  INFO      Columns: ['Age', 'Age_sq', 'Age_x_Wealth', 'FinancialEducation', 'RiskPropensity', 'FinEdu_x_Risk', 'Income_log', 'Wealth_log', 'Income_per_FM_log', 'Wealth_per_FM_log']


,Age,Age_sq,Age_x_Wealth,FinancialEducation,RiskPropensity,FinEdu_x_Risk,Income_log,Wealth_log,Income_per_FM_log,Wealth_per_FM_log
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,55.253400,3196.231000,236.068108,0.419123,0.362686,0.167640,3.902163,4.176349,3.072326,3.339914
std,11.971694,1346.463673,86.552449,0.151383,0.151124,0.115577,0.771960,0.888836,0.748095,0.855311
min,18.000000,324.000000,14.424221,0.036099,0.024790,0.001280,0.931284,0.721450,0.395465,0.267837
25%,47.000000,2209.000000,178.130702,0.308067,0.246444,0.079914,3.453046,3.671507,2.632376,2.827673
50%,55.000000,3025.000000,228.976200,0.416735,0.354526,0.143239,3.996352,4.205746,3.118531,3.344498
75%,63.000000,3969.000000,285.905809,0.523393,0.467125,0.228820,4.444089,4.752078,3.569168,3.883239
max,97.000000,9409.000000,740.318518,0.902933,0.882709,0.797027,5.903517,7.711651,5.674849,6.942683


In [36]:
assert list(X.columns) == FEATURE_NAMES, f"Column mismatch: {list(X.columns)}"
assert X.shape == (5000, len(FEATURE_NAMES)), f"Wrong shape: {X.shape}"
assert X.isnull().sum().sum() == 0, "NaN values in engineered features"

# Formula spot-checks
assert np.allclose(X['Age_sq'], df['Age'] ** 2), "Age_sq formula wrong"
assert np.allclose(X['FinEdu_x_Risk'], df['FinancialEducation'] * df['RiskPropensity']), \
    "FinEdu_x_Risk formula wrong"
assert np.allclose(X['Income_log'], np.log1p(df['Income'])), "Income_log formula wrong"
assert np.allclose(X['Wealth_log'], np.log1p(df['Wealth'])), "Wealth_log formula wrong"
assert np.allclose(X['Age_x_Wealth'], df['Age'] * np.log1p(df['Wealth'])), \
    "Age_x_Wealth formula wrong — should be Age × log1p(Wealth)"
assert np.allclose(X['Income_per_FM_log'], np.log1p(df['Income'] / df['FamilyMembers'])), \
    "Income_per_FM_log formula wrong"
assert np.allclose(X['Wealth_per_FM_log'], np.log1p(df['Wealth'] / df['FamilyMembers'])), \
    "Wealth_per_FM_log formula wrong"

# Excluded variables must be absent
assert 'Gender' not in X.columns, \
    "Gender must be excluded — r < 0.015 with both targets, pure noise"
assert 'FamilyMembers' not in X.columns, \
    "Raw FamilyMembers must be excluded — superseded by per-capita ratio features"

logger.info("Engineered feature checks PASSED")

14:51:27  INFO      Engineered feature checks PASSED


## 5. No-leakage check — scaler fitted on train only

In [37]:
for target in TARGETS:
    X_tr, X_te, y_tr, y_te, scaler = split_and_scale(X, df[target])

    assert scaler.data_min_ is not None, "Scaler not fitted"
    assert X_tr.min().min() >= -1e-9, "Train set minimum below 0 (scaling bug)"
    assert X_tr.max().max() <= 1.0 + 1e-9, "Train set maximum above 1 (scaling bug)"

    logger.info(
        "%s — train range [%.4f, %.4f] | test range [%.4f, %.4f] (can exceed [0,1] — correct)",
        target,
        X_tr.min().min(), X_tr.max().max(),
        X_te.min().min(), X_te.max().max(),
    )

logger.info("Section 5 PASSED — no-leakage check confirmed for all targets")

14:51:44  INFO      IncomeInvestment — train range [0.0000, 1.0000] | test range [-0.0128, 1.0811] (can exceed [0,1] — correct)


14:51:44  INFO      AccumulationInvestment — train range [0.0000, 1.0000] | test range [-0.0156, 1.1178] (can exceed [0,1] — correct)
14:51:44  INFO      Section 5 PASSED — no-leakage check confirmed for all targets


## 6. Stratification — class balance preserved across split

In [38]:
for target in TARGETS:
    X_tr, X_te, y_tr, y_te, scaler = split_and_scale(X, df[target])
    train_prev = y_tr.mean()
    test_prev  = y_te.mean()
    delta = abs(train_prev - test_prev)
    print(f"{target}: train={train_prev:.4f}  test={test_prev:.4f}  delta={delta:.4f}")
    assert delta < 0.03, \
        f"{target}: stratification drift too large: delta={delta:.4f}"

logging.info("Stratification check PASSED")

14:51:53  INFO      Stratification check PASSED


IncomeInvestment: train=0.3835  test=0.3840  delta=0.0005
AccumulationInvestment: train=0.5132  test=0.5130  delta=0.0002


## 7. Correlation sanity — engineered features vs targets

In [39]:

full = pd.concat([X, df[TARGETS]], axis=1)
corr = full.corr()

checks = [
    # (label, value, operator, threshold)
    # --- Feature vs target correlations (from paper EDA section) ---
    ("Age_x_Wealth  vs IncomeInvestment",
     corr.loc['Age_x_Wealth', 'IncomeInvestment'],      '>', 0.30),
    ("Age           vs IncomeInvestment",
     corr.loc['Age', 'IncomeInvestment'],                '>', 0.25),
    ("Wealth_log    vs IncomeInvestment",
     corr.loc['Wealth_log', 'IncomeInvestment'],         '>', 0.30),
    ("Income_log    vs AccumulationInvestment",
     corr.loc['Income_log', 'AccumulationInvestment'],   '>', 0.25),
    # --- Multicollinearity — expected by construction, not problematic ---
    ("FinEdu        vs RiskPropensity",
     corr.loc['FinancialEducation', 'RiskPropensity'],   '>', 0.60),
    ("Income_per_FM vs Income_log",
     corr.loc['Income_per_FM_log', 'Income_log'],        '>', 0.85),
    # --- Target near-independence ---
    ("IncomeInvestment vs AccumulationInvestment",
     abs(corr.loc['IncomeInvestment', 'AccumulationInvestment']), '<', 0.05),
]

for label, value, op, threshold in checks:
    passed = (value > threshold) if op == '>' else (value < threshold)
    status = "PASS" if passed else "FAIL"
    logging.info(f"[{status}]  {label}: {value:.3f}  ({op} {threshold})")
    assert passed, f"Correlation check failed: {label} = {value:.3f}, expected {op} {threshold}"

logging.info("\nCorrelation sanity checks PASSED")

14:52:05  INFO      [PASS]  Age_x_Wealth  vs IncomeInvestment: 0.446  (> 0.3)
14:52:05  INFO      [PASS]  Age           vs IncomeInvestment: 0.334  (> 0.25)
14:52:05  INFO      [PASS]  Wealth_log    vs IncomeInvestment: 0.404  (> 0.3)
14:52:05  INFO      [PASS]  Income_log    vs AccumulationInvestment: 0.322  (> 0.25)
14:52:05  INFO      [PASS]  FinEdu        vs RiskPropensity: 0.683  (> 0.6)
14:52:05  INFO      [PASS]  Income_per_FM vs Income_log: 0.908  (> 0.85)
14:52:05  INFO      [PASS]  IncomeInvestment vs AccumulationInvestment: 0.011  (< 0.05)
14:52:05  INFO      
Correlation sanity checks PASSED


## 8. Raw vs engineered feature comparison

In [40]:
X_base = build_baseline_features(df)

logger.info("Engineered features (F_E): %s", list(X.columns))
logger.info("Baseline features    (F_B): %s", list(X_base.columns))

added = set(FEATURE_NAMES) - set(BASELINE_FEATURE_NAMES)
logger.info("Features added by engineering: %s", sorted(added))

assert 'Gender' not in FEATURE_NAMES, \
    "Gender must be excluded from engineered set — r < 0.015 with both targets, pure noise"
assert 'Gender' in BASELINE_FEATURE_NAMES, \
    "Gender should be present in baseline set (professor's original feature set)"
assert 'FamilyMembers' not in FEATURE_NAMES, \
    "Raw FamilyMembers must not appear in the engineered set"
assert set(BASELINE_FEATURE_NAMES).issubset(set(X_base.columns)), \
    "Baseline feature set missing expected columns"
assert set(FEATURE_NAMES) == set(X.columns), \
    "Engineered feature set columns do not match FEATURE_NAMES constant"
assert len(added) > 0, \
    "No features were added by engineering — check build_features()"

logger.info("Section 8 PASSED — feature set structure confirmed, %d features added by engineering", len(added))

14:52:25  INFO      Engineered features (F_E): ['Age', 'Age_sq', 'Age_x_Wealth', 'FinancialEducation', 'RiskPropensity', 'FinEdu_x_Risk', 'Income_log', 'Wealth_log', 'Income_per_FM_log', 'Wealth_per_FM_log']
14:52:25  INFO      Baseline features    (F_B): ['Age', 'Gender', 'FamilyMembers', 'FinancialEducation', 'RiskPropensity', 'Income_log', 'Wealth_log']
14:52:25  INFO      Features added by engineering: ['Age_sq', 'Age_x_Wealth', 'FinEdu_x_Risk', 'Income_per_FM_log', 'Wealth_per_FM_log']
14:52:25  INFO      Section 8 PASSED — feature set structure confirmed, 5 features added by engineering


## 9. Pickle directory status

In [41]:
PICKLE_ROOT = Path('data/pickled_files')
model_folders = [
    'linear_reg', 'naive_bayes', 'rand_forest',
    'xgboost_shap', 'mlp', 'classifier_chain',
    'soft_voting_ens', 'hard_voting_ens',
]

logging.info("Model output status:")
for folder in model_folders:
    p = PICKLE_ROOT / folder
    if p.exists():
        files = list(p.glob('*.joblib'))
        status = f"{len(files)} joblib file(s): {[f.name for f in files]}"
    else:
        status = "directory not found"
    logging.info(f"  {folder:25s} → {status}")

14:52:38  INFO      Model output status:
14:52:38  INFO        linear_reg                → 0 joblib file(s): []
14:52:38  INFO        naive_bayes               → 0 joblib file(s): []
14:52:38  INFO        rand_forest               → 0 joblib file(s): []
14:52:38  INFO        xgboost_shap              → 0 joblib file(s): []
14:52:38  INFO        mlp                       → 0 joblib file(s): []
14:52:38  INFO        classifier_chain          → 0 joblib file(s): []
14:52:38  INFO        soft_voting_ens           → 0 joblib file(s): []
14:52:38  INFO        hard_voting_ens           → 0 joblib file(s): []


## Summary

All assertions passed:
1. Raw data: 5,000 rows, 9 columns, 0 nulls, `'Income '` trailing-space fixed
2. Class balance: IncomeInvestment 38.4% positive (1,918/3,082), AccumulationInvestment 51.3% positive (2,566/2,434)
3. Skewness: Wealth = 5.83 (threshold > 2.0), Income = 1.38 (threshold > 1.0) → log transforms justified
4. Engineered features (F_E): 10 columns, 0 nulls, all formula spot-checks passed
   - Added vs baseline: `Age_sq`, `Age_x_Wealth`, `FinEdu_x_Risk`, `Income_per_FM_log`, `Wealth_per_FM_log`
   - Excluded vs baseline: `Gender` (r < 0.015), `FamilyMembers` raw (superseded by per-capita ratios)
5. No-leakage: scaler fitted on train only — train range [0.0, 1.0], test range [-0.013, 1.081] (expected, correct)
6. Stratification: both targets preserved across split (IncomeInvestment δ = 0.0005, AccumulationInvestment δ = 0.0002)
7. Correlations — all 7 paper claims confirmed:
   - `Age_x_Wealth` vs IncomeInvestment: r = 0.446 (> 0.30) ✓
   - `Age` vs IncomeInvestment: r = 0.334 (> 0.25) ✓
   - `Wealth_log` vs IncomeInvestment: r = 0.404 (> 0.30) ✓
   - `Income_log` vs AccumulationInvestment: r = 0.322 (> 0.25) ✓
   - `FinancialEducation` vs `RiskPropensity`: r = 0.683 (> 0.60) — collinearity expected ✓
   - `Income_per_FM_log` vs `Income_log`: r = 0.908 (> 0.85, < 1.0) — FamilyMembers adds genuine signal ✓
   - IncomeInvestment vs AccumulationInvestment: r = 0.011 (< 0.05) — targets near-independent ✓
8. Feature sets: F_E (10 features) and F_B (7 features) structure validated — no unexpected columns
9. Model outputs: all 8 model directories exist, no trained files yet (training not yet run)

## 10. Persist feature matrices

Saves F_E, F_B, and targets to disk so all model notebooks load
from a single source of truth. Feature construction runs exactly once.
Scaling happens inside each model's Pipeline — never here.

In [43]:
from pathlib import Path
import pickle

FEATURE_STORE = Path('data/feature_store')
FEATURE_STORE.mkdir(parents=True, exist_ok=True)

with open(FEATURE_STORE / 'F_E.pkl', 'wb') as f:
    pickle.dump(X, f)

with open(FEATURE_STORE / 'F_B.pkl', 'wb') as f:
    pickle.dump(X_base, f)

with open(FEATURE_STORE / 'targets.pkl', 'wb') as f:
    pickle.dump(df[TARGETS], f)

logger.info("Saved F_E     → %s  shape=%s", FEATURE_STORE / 'F_E.pkl',     X.shape)
logger.info("Saved F_B     → %s  shape=%s", FEATURE_STORE / 'F_B.pkl',     X_base.shape)
logger.info("Saved targets → %s  shape=%s", FEATURE_STORE / 'targets.pkl', df[TARGETS].shape)
logger.info("Feature store ready — models load from here, never call build_features() directly")

14:55:01  INFO      Saved F_E     → data\feature_store\F_E.pkl  shape=(5000, 10)
14:55:01  INFO      Saved F_B     → data\feature_store\F_B.pkl  shape=(5000, 7)
14:55:01  INFO      Saved targets → data\feature_store\targets.pkl  shape=(5000, 2)


14:55:01  INFO      Feature store ready — models load from here, never call build_features() directly
